In [2]:
import joblib
import pandas as pd

In [10]:
df = pd.read_excel("AcademiX_Grade11_ICT_Dataset.xlsx", sheet_name="AcademiX_Dataset")
df.head()

,Student_ID,Student_Name,Lesson_ID,Lesson_Name,Module_1_Score,Module_2_Score,Module_3_Score,Avg_Module_Score,Followup_Quiz_Score,Term_Test_Marks_Obtained
0,ST001,Nimal Perera,L1,Information and Communication Technology,21.25,23.00,21.50,21.92,22.42,13.83
1,ST001,Nimal Perera,L2,Fundamentals of a Computer System,21.25,23.75,20.75,21.92,22.67,13.00
2,ST001,Nimal Perera,L3,Data Representation Methods in Computer Systems,20.25,24.00,22.00,22.08,23.58,7.04
3,ST001,Nimal Perera,L4,Logic Gates with Boolean Functions,21.00,23.50,25.00,23.17,24.92,7.82
4,ST001,Nimal Perera,L5,Operating Systems,22.50,20.75,21.25,21.50,21.50,13.07


In [11]:
df.isnull().sum()

Student_ID                  0
Student_Name                0
Lesson_ID                   0
Lesson_Name                 0
Module_1_Score              0
Module_2_Score              0
Module_3_Score              0
Avg_Module_Score            0
Followup_Quiz_Score         0
Term_Test_Marks_Obtained    0
dtype: int64

In [12]:
df.shape

(531, 10)

In [13]:
df_encoded = pd.get_dummies(df, columns=['Lesson_ID'], prefix='LessonID')
df_encoded.head()

,Student_ID,Student_Name,Lesson_Name,Module_1_Score,Module_2_Score,Module_3_Score,Avg_Module_Score,Followup_Quiz_Score,Term_Test_Marks_Obtained,LessonID_L1,LessonID_L2,LessonID_L3,LessonID_L4,LessonID_L5,LessonID_L6,LessonID_L7,LessonID_L8,LessonID_L9
0,ST001,Nimal Perera,Information and Communication Technology,21.25,23.00,21.50,21.92,22.42,13.83,True,False,False,False,False,False,False,False,False
1,ST001,Nimal Perera,Fundamentals of a Computer System,21.25,23.75,20.75,21.92,22.67,13.00,False,True,False,False,False,False,False,False,False
2,ST001,Nimal Perera,Data Representation Methods in Computer Systems,20.25,24.00,22.00,22.08,23.58,7.04,False,False,True,False,False,False,False,False,False
3,ST001,Nimal Perera,Logic Gates with Boolean Functions,21.00,23.50,25.00,23.17,24.92,7.82,False,False,False,True,False,False,False,False,False
4,ST001,Nimal Perera,Operating Systems,22.50,20.75,21.25,21.50,21.50,13.07,False,False,False,False,True,False,False,False,False


In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

base_features = ['Module_1_Score','Module_2_Score','Module_3_Score','Avg_Module_Score','Followup_Quiz_Score']
lesson_cols = [c for c in df_encoded.columns if c.startswith('LessonID_')]
features = base_features + lesson_cols

X = df_encoded[features]
y = df_encoded['Term_Test_Marks_Obtained']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

y_predict = model.predict(X_test)
print("r2 score:", r2_score(y_test, y_predict))
print("MAE:", mean_absolute_error(y_test, y_predict))
print("RMSE:", mean_squared_error(y_test, y_predict) ** 0.5)

r2 score: 0.9816279145100196
MAE: 0.35328224299065397
RMSE: 0.44674293659986275


In [17]:
joblib.dump(model, "term_score_predictor.pkl")
joblib.dump(features, "model_features.pkl")  # save feature order/columns for future predictions

print("Model saved successfully!")

Model saved successfully!


In [18]:
loaded_model = joblib.load("term_score_predictor.pkl")
loaded_features = joblib.load("model_features.pkl")

# example: Module scores (out of 25) + Followup quiz (out of 25) for a student on Lesson L1
sample = pd.DataFrame([{
    'Module_1_Score': 21.25,
    'Module_2_Score': 23,
    'Module_3_Score': 21.5,
    'Avg_Module_Score': 21.92,
    'Followup_Quiz_Score': 22.42,
    'LessonID_L1': 1,
    'LessonID_L2': 0,
    'LessonID_L3': 0,
    'LessonID_L4': 0,
    'LessonID_L5': 0,
    'LessonID_L6': 0,
    'LessonID_L7': 0,
    'LessonID_L8': 0,
    'LessonID_L9': 0,
}])[loaded_features]

prediction = loaded_model.predict(sample)
print(prediction)

[13.4735]
